# Adaptive Learning Rate — Exercises

8 exercises covering AdaGrad, RMSProp, Adam bias correction, AdamW, LAMB, Adafactor, and optimizer diagnostics.

| Format | Description |
| --- | --- |
| **Problem** | Markdown cell with task description |
| **Your Solution** | Runnable scaffold with placeholders |
| **Solution** | Reference solution with checks |

### Difficulty Levels

| Level | Exercises | Focus |
| --- | --- | --- |
| ★ | 1-3 | Core mechanics |
| ★★ | 4-6 | Adam family and layerwise theory |
| ★★★ | 7-8 | Memory and diagnostics |


In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print("  expected:", expected)
        print("  got     :", got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

print("Exercise setup complete.")


---

## Exercise 1 ★ — AdaGrad by Hand

Compute AdaGrad accumulators and updates for scalar gradients.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 1: AdaGrad by Hand
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 1: AdaGrad by Hand")
grads = np.array([1.0, 2.0, 2.0])
r = 0.0
steps = []
theta = 0.0
for g in grads:
    r += g*g
    step = 0.1 * g / np.sqrt(r)
    theta -= step
    steps.append(step)
print("accumulator:", r)
print("steps:", steps)
print("theta:", theta)
check_close("final accumulator", r, 9.0)
check_true("steps shrink relative to raw gradient", steps[-1] < 0.1 * grads[-1])
print("\nTakeaway: AdaGrad remembers all squared gradients, so frequent coordinates receive smaller effective steps.")


---

## Exercise 2 ★ — RMSProp Forgetting

Track exponential second moments after a gradient scale change.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 2: RMSProp Forgetting
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 2: RMSProp Forgetting")
grads = np.r_[np.ones(5), 5*np.ones(5), np.ones(10)]
r = 0.0
track = []
for g in grads:
    r = 0.9*r + 0.1*g*g
    track.append(r)
print("tracked second moments:", np.round(track, 4))
check_true("spike increases second moment", track[9] > track[4])
check_true("forgetting lowers it after spike", track[-1] < track[9])
print("\nTakeaway: RMSProp keeps recent scale information without AdaGrad's unbounded accumulation.")


---

## Exercise 3 ★ — Adam Bias Correction

Verify corrected moments for a constant gradient.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 3: Adam Bias Correction
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 3: Adam Bias Correction")
g = 4.0
b1, b2 = 0.9, 0.999
m = v = 0.0
for t in range(1, 4):
    m = b1*m + (1-b1)*g
    v = b2*v + (1-b2)*g*g
mhat = m / (1-b1**t)
vhat = v / (1-b2**t)
print("mhat:", mhat)
print("vhat:", vhat)
check_close("corrected first moment", mhat, g)
check_close("corrected second moment", vhat, g*g)
print("\nTakeaway: bias correction removes the zero-initialization bias for constant gradients.")


---

## Exercise 4 ★★ — Effective Learning Rates

Compare coordinatewise effective learning rates.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 4: Effective Learning Rates
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 4: Effective Learning Rates")
v = np.array([1e-4, 1e-2, 1.0, 100.0])
eff = 1e-3 / (np.sqrt(v) + 1e-8)
print("effective learning rates:", eff)
check_true("effective LR decreases with second moment", np.all(np.diff(eff) < 0))
check_close("largest effective LR", eff[0], 0.1, tol=1e-5)
print("\nTakeaway: adaptive optimizers still have learning rates; they are coordinatewise effective learning rates.")


---

## Exercise 5 ★★ — AdamW vs Coupled L2

Show coupled and decoupled decay differ.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 5: AdamW vs Coupled L2
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 5: AdamW vs Coupled L2")
theta = np.array([1.0, 10.0])
g = np.array([0.1, 0.1])
v = np.array([1e-4, 1.0])
lr, wd = 0.01, 0.1
coupled = theta - lr * (g + wd*theta) / (np.sqrt(v) + 1e-8)
decoupled = (1 - lr*wd) * theta - lr * g / (np.sqrt(v) + 1e-8)
print("coupled:", coupled)
print("decoupled:", decoupled)
check_true("updates differ", not np.allclose(coupled, decoupled))
print("\nTakeaway: AdamW decouples weight decay from adaptive gradient preconditioning.")


---

## Exercise 6 ★★ — LAMB Trust Ratio

Compute layerwise trust ratios.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 6: LAMB Trust Ratio
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 6: LAMB Trust Ratio")
theta = np.array([3.0, 4.0])
u = np.array([0.3, 0.4])
trust = la.norm(theta) / (la.norm(u) + 1e-8)
scaled_norm = la.norm(trust * u)
print("trust ratio:", trust)
print("scaled update norm:", scaled_norm)
check_close("trust ratio", trust, 10.0, tol=1e-6)
check_close("scaled update norm matches parameter norm", scaled_norm, la.norm(theta), tol=1e-6)
print("\nTakeaway: trust ratios control layerwise update scale relative to parameter scale.")


---

## Exercise 7 ★★★ — Adafactor Memory

Compare full and factored second-moment memory.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 7: Adafactor Memory
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 7: Adafactor Memory")
m, n = 4096, 16384
full = m * n
factored = m + n
savings = full / factored
print("full entries:", full)
print("factored entries:", factored)
print("savings factor:", savings)
check_true("factored state is much smaller", savings > 1000)
print("\nTakeaway: factored second moments can reduce optimizer state from O(mn) to O(m+n).")


---

## Exercise 8 ★★★ — Optimizer Diagnostics

Diagnose a synthetic failed training run.

Fill the scaffold, then compare with the reference solution.

In [ ]:
# Your Solution — Exercise 8: Optimizer Diagnostics
# Replace None with your computation.
answer = None
print("answer:", answer)


In [ ]:
header("Exercise 8: Optimizer Diagnostics")
grad_norm = np.array([1.0, 1.2, 1.4, 20.0, 25.0])
update_ratio = np.array([0.001, 0.0012, 0.0013, 0.08, 0.12])
effective_lr = np.array([1e-3, 1e-3, 1e-3, 5e-2, 8e-2])
failure = (grad_norm[-1] > 10 * np.median(grad_norm[:3])) and (update_ratio[-1] > 0.05)
print("failure detected:", failure)
print("suggested fix: lower LR, inspect loss scaling, add clipping or increase epsilon")
check_true("synthetic run shows optimizer instability", failure)
print("\nTakeaway: gradient norm, update ratio, and effective LR often expose failures before loss alone explains them.")
